In [ ]:
#@title Prevent disconnections
%%html
<audio src="https://oobabooga.github.io/silence.m4a" controls>

In [ ]:
#@title Setup SwarmUI
import os
SWARMPATH = '/content/'
os.environ['SWARMPATH'] = SWARMPATH
os.environ['SWARM_NO_VENV'] = 'true'

!apt install -y aria2

# Install dotnet 8.0
!wget -q https://dot.net/v1/dotnet-install.sh -O dotnet-install.sh
!chmod +x dotnet-install.sh
!./dotnet-install.sh --channel 8.0

# Install cloudflared for sharing
!wget -q https://github.com/cloudflare/cloudflared/releases/download/2024.8.2/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

%cd $SWARMPATH

# Clone SwarmUI
!git clone https://github.com/mcmonkeyprojects/SwarmUI.git

# Create model directory
!mkdir -p /content/SwarmUI/Models/checkpoints

In [ ]:
#@title Select & Download Model
KEY = ""  # @param {type:"string"}
KEY = "token=" + KEY.strip()

MODELS = {
    "mklan_hentai_223": {
        "file": "mklan_hentai_223.safetensors",
        "url": f"https://civitai.com/api/download/models/439047?{KEY}"
    },
    "mklanANIMEHentai_nextgenV4": {
        "file": "mklanANIMEHentai_nextgenV4.safetensors",
        "url": f"https://civitai.com/api/download/models/812350?type=Model&format=SafeTensor&size=pruned&fp=fp16&{KEY}"
    },
    "deepDarkHentaiMixNSFW_v61Hybrid": {
        "file": "deepDarkHentaiMixNSFW_v61Hybrid.safetensors",
        "url": f"https://civitai.com/api/download/models/634653?type=Model&format=SafeTensor&size=pruned&fp=fp16&{KEY}"
    },
    "cyberrealisticPony_semiRealV40": {
        "file": "cyberrealisticPony_semiRealV40.safetensors",
        "url": f"https://civitai.com/api/download/models/2268768?type=Model&format=SafeTensor&size=pruned&fp=fp16&{KEY}"
    },
    "zyntoonSemiRealistic_v10MainVAE": {
        "file": "zyntoonSemiRealistic_v10MainVAE.safetensors",
        "url": f"https://civitai.com/api/download/models/959032?type=Model&format=SafeTensor&size=full&fp=fp16&{KEY}"
    },
    "dragonslayerstavern_v19ETDDtsv1Merge": {
        "file": "dragonslayerstavern_v19ETDDtsv1Merge.safetensors",
        "url": f"https://civitai.com/api/download/models/1419328?type=Model&format=SafeTensor&size=full&fp=fp16&{KEY}"
    },
    "novaCartoon_v10": {
        "file": "novaCartoon_v10.safetensors",
        "url": f"https://civitai.com/api/download/models/821389?type=Model&format=SafeTensor&size=pruned&fp=fp16&{KEY}"
    },
}

CHOICE = "deepDarkHentaiMixNSFW_v61Hybrid"  # @param ["mklan_hentai_223", "mklanANIMEHentai_nextgenV4", "deepDarkHentaiMixNSFW_v61Hybrid", "cyberrealisticPony_semiRealV40", "zyntoonSemiRealistic_v10MainVAE", "dragonslayerstavern_v19ETDDtsv1Merge", "novaCartoon_v10"]

MODEL = MODELS[CHOICE]
MODEL_PATH = "/content/SwarmUI/Models/checkpoints"
USER_AGENT = '"User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64)"'

!aria2c --enable-http-keep-alive=false --header=$USER_AGENT --console-log-level=error -c -x 16 -s 16 -k 1M --summary-interval=5 -d $MODEL_PATH -o {MODEL['file']} "{MODEL['url']}"

In [ ]:
#@title Launch SwarmUI
%cd /content/SwarmUI

!git fetch
!git reset --hard origin/master
!git pull --autostash
!rm -rf ./src/bin/live_release

!bash ./launch-linux.sh --launch_mode none --cloudflared-path cloudflared